# TakaSecure Banking SFT — RunPod QLoRA

This notebook fine-tunes a Llama 3.1 8B Instruct model on the 7,000-row synthetic banking dataset. It uses policy-family-isolated train/validation/test splits, response-only loss, deterministic generation, citation checks, and LoRA adapter export.

All policies and cases are fictional. This notebook is for model-behavior research and is not a source of banking or regulatory advice.

Recommended RunPod hardware: one RTX 3090/4090, A5000, or A10 with 24 GB VRAM. A T4 16 GB can work more slowly with the default batch size.

## 1. RunPod preparation

Open this notebook from the cloned repository under `/workspace`. If the repository is not present, run the following in a RunPod terminal first:

```bash
cd /workspace
git clone https://github.com/Shoaib-33/TakaSecure-AI-based-Secure-Banking.git
```

Optionally configure `HF_TOKEN` and `HF_MODEL_ID` as RunPod secrets/environment variables before starting the Pod.

In [ ]:
import os
from pathlib import Path

# Keep model caches and training artifacts on RunPod's persistent volume.
os.environ.setdefault("HF_HOME", "/workspace/.cache/huggingface")
os.environ.setdefault("TORCH_HOME", "/workspace/.cache/torch")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path("/workspace/TakaSecure-AI-based-Secure-Banking")]
    for candidate in candidates:
        if (candidate / "generate_banking_sft.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Could not find TakaSecure AI. Clone it under /workspace or open this notebook from the repository.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "generated"
OUTPUT_DIR = Path("/workspace/takasecure-banking-sft-v3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
SEED = 42
EPOCHS = 1.0
LEARNING_RATE = 5e-5
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
EVAL_STEPS = 50
GENERATION_EVAL_SAMPLES = 100
HF_MODEL_ID = os.getenv("HF_MODEL_ID", "")  # Example: username/takasecure-llama-lora
SAVE_MERGED_16BIT = False  # Adapter-only is smaller and works with vLLM LoRA serving.

print("Project:", PROJECT_ROOT)
print("Output:", OUTPUT_DIR)
print("Push target:", HF_MODEL_ID or "disabled")

## 2. Install the pinned training environment

Run this in a fresh RunPod PyTorch notebook kernel. If Unsloth, Transformers, or TRL were already imported earlier in the kernel, restart the kernel after installation and then continue from the configuration cell.

In [ ]:
%pip install -q --upgrade "unsloth==2026.7.2"

In [ ]:
import importlib.metadata
import json
import random
import re
import shutil
import subprocess
import sys

import numpy as np
import torch
from datasets import Dataset
from transformers import EarlyStoppingCallback, set_seed

# Unsloth must be imported before trainer classes so its patches are applied.
from unsloth import FastLanguageModel, UnslothTrainer, UnslothTrainingArguments, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not detected. Deploy a RunPod GPU Pod before training.")

random.seed(SEED)
np.random.seed(SEED)
set_seed(SEED)

packages = {}
for package in ("unsloth", "unsloth-zoo", "transformers", "trl", "datasets", "torch"):
    try:
        packages[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        packages[package] = "not-installed"

gpu = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(gpu.total_memory / 1024**3, 2))
print("Packages:", json.dumps(packages, indent=2))

## 3. Generate and verify the dataset

The generated JSONL files are intentionally reproducible. This cell regenerates them from the versioned policy catalog and runs the generator's quality gates.

In [ ]:
generator = PROJECT_ROOT / "generate_banking_sft.py"
subprocess.run([sys.executable, str(generator), "--output-dir", str(DATA_DIR)], check=True)

quality_report = json.loads((DATA_DIR / "quality_report.json").read_text(encoding="utf-8"))
assert quality_report["valid"] is True
assert quality_report["total_rows"] == 7000
assert all(value == 0 for value in quality_report["policy_overlap"].values())
quality_report

In [ ]:
def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

train_rows = load_jsonl(DATA_DIR / "banking_sft_train.jsonl")
validation_rows = load_jsonl(DATA_DIR / "banking_sft_validation.jsonl")
test_rows = load_jsonl(DATA_DIR / "banking_sft_test.jsonl")

assert len(train_rows) == 5600
assert len(validation_rows) == 700
assert len(test_rows) == 700

print({"train": len(train_rows), "validation": len(validation_rows), "test": len(test_rows)})
print(json.dumps(train_rows[0], indent=2, ensure_ascii=False)[:3000])

## 4. Load the 4-bit base model and attach LoRA

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

## 5. Apply the model's chat template and filter overlength examples

Only the `messages` field is used for SFT. Metadata remains in the JSONL files for auditing and evaluation.

In [ ]:
def to_hf_dataset(rows):
    return Dataset.from_list([{"messages": row["messages"]} for row in rows])

def apply_chat_template(batch):
    return {
        "text": [
            tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            for messages in batch["messages"]
        ]
    }

def add_token_length(batch):
    encoded = tokenizer(batch["text"], add_special_tokens=False, truncation=False)
    return {"token_length": [len(ids) for ids in encoded["input_ids"]]}

def prepare(rows, split_name):
    dataset = to_hf_dataset(rows)
    dataset = dataset.map(apply_chat_template, batched=True, batch_size=128, remove_columns=["messages"], desc=f"Formatting {split_name}")
    dataset = dataset.map(add_token_length, batched=True, batch_size=128, desc=f"Measuring {split_name}")
    before = len(dataset)
    dataset = dataset.filter(lambda row: row["token_length"] <= MAX_SEQ_LENGTH, desc=f"Filtering {split_name}")
    print(f"{split_name}: kept {len(dataset)}/{before}; removed {before - len(dataset)} overlength rows")
    return dataset

train_dataset = prepare(train_rows, "train")
validation_dataset = prepare(validation_rows, "validation")
test_dataset = prepare(test_rows, "test")

print(train_dataset[0]["text"][:1500])

## 6. Configure response-only supervised fine-tuning

The instruction and context tokens are masked from loss. Training targets only assistant responses.

In [ ]:
trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=4,
    packing=False,
    args=UnslothTrainingArguments(
        output_dir=str(OUTPUT_DIR / "checkpoints"),
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=EPOCHS,
        warmup_ratio=0.03,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy="steps",
        save_steps=EVAL_STEPS,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        seed=SEED,
        data_seed=SEED,
        report_to="none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=2))

effective_batch_size = TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
print("Effective batch size:", effective_batch_size)
print("Training examples:", len(train_dataset))

## 7. Train

This is the long-running cell. Run it inside a persistent RunPod Pod and keep the output directory under `/workspace`.

In [ ]:
train_result = trainer.train()
train_result.metrics

## 8. Evaluate validation and held-out policy families

In [ ]:
response_marker = "<|start_header_id|>assistant<|end_header_id|>\n\n"
response_marker_ids = tokenizer(response_marker, add_special_tokens=False)["input_ids"]

def find_last_marker(sequence, marker):
    for index in range(len(sequence) - len(marker), -1, -1):
        if sequence[index:index + len(marker)] == marker:
            return index
    return None

def tokenize_response_only(batch):
    encoded = tokenizer(
        batch["text"],
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    labels = []
    for input_ids in encoded["input_ids"]:
        marker_start = find_last_marker(input_ids, response_marker_ids)
        if marker_start is None:
            raise ValueError("Assistant response marker was not found")
        response_start = marker_start + len(response_marker_ids)
        example_labels = input_ids.copy()
        example_labels[:response_start] = [-100] * response_start
        labels.append(example_labels)
    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": labels,
    }

def prepare_evaluation_dataset(dataset, split_name):
    prepared = dataset.map(
        tokenize_response_only,
        batched=True,
        batch_size=128,
        remove_columns=dataset.column_names,
        load_from_cache_file=False,
        desc=f"Tokenizing {split_name}",
    )
    required = {"input_ids", "attention_mask", "labels"}
    if not required.issubset(prepared.column_names):
        raise RuntimeError(f"{split_name} dataset was not tokenized correctly")
    return prepared

tokenized_validation = prepare_evaluation_dataset(validation_dataset, "validation")
tokenized_test = prepare_evaluation_dataset(test_dataset, "test")
validation_metrics = trainer.evaluate(eval_dataset=tokenized_validation, metric_key_prefix="validation")
test_metrics = trainer.evaluate(eval_dataset=tokenized_test, metric_key_prefix="test")
metrics = {"train": train_result.metrics, "validation": validation_metrics, "test": test_metrics}
(OUTPUT_DIR / "trainer_metrics.json").write_text(json.dumps(metrics, indent=2, default=str) + "\n", encoding="utf-8")
metrics


## 9. Save the LoRA adapter

In [ ]:
ADAPTER_DIR = OUTPUT_DIR / "adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

run_report = {
    "base_model": MODEL_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "effective_batch_size": effective_batch_size,
    "seed": SEED,
    "train_rows": len(train_dataset),
    "validation_rows": len(validation_dataset),
    "test_rows": len(test_dataset),
    "response_only_loss": True,
    "dataset_quality": quality_report,
    "packages": packages,
    "gpu": torch.cuda.get_device_name(0),
}
(OUTPUT_DIR / "run_report.json").write_text(json.dumps(run_report, indent=2, default=str) + "\n", encoding="utf-8")
print("Saved adapter to", ADAPTER_DIR)

## 10. Correct deterministic inference

Only newly generated tokens are decoded. An attention mask is passed explicitly.

In [ ]:
FastLanguageModel.for_inference(model)

def generate_response(messages, max_new_tokens=384):
    prompt_messages = [message for message in messages if message["role"] != "assistant"]
    input_ids = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(next(model.parameters()).device)
    attention_mask = torch.ones_like(input_ids)
    with torch.inference_mode():
        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0, input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

sample = test_rows[0]
prediction = generate_response(sample["messages"])
print("TASK:", sample["task_type"])
print("EXPECTED:", sample["messages"][2]["content"])
print("PREDICTED:", prediction)

## 11. Generation-level structure and citation evaluation

This is a fast behavioral check, not a substitute for expert review. It evaluates deterministic samples from the held-out policy-family test split.

In [ ]:
eval_rng = random.Random(SEED)
sample_count = min(GENERATION_EVAL_SAMPLES, len(test_rows))
evaluation_rows = [test_rows[index] for index in eval_rng.sample(range(len(test_rows)), sample_count)]
citation_pattern = re.compile(r"TSB-[A-Z-]+-\d{2}-\d(?:-LEGACY)?")
results = []

for position, row in enumerate(evaluation_rows, start=1):
    prediction = generate_response(row["messages"])
    expected = set(row["expected_citations"])
    allowed = set(row["context_policy_ids"])
    found = set(citation_pattern.findall(prediction))
    json_valid = None
    if row["response_format"] == "json":
        try:
            json.loads(prediction)
            json_valid = True
        except json.JSONDecodeError:
            json_valid = False
    results.append({
        "id": row["id"],
        "task_type": row["task_type"],
        "response_format": row["response_format"],
        "expected_citations": sorted(expected),
        "found_citations": sorted(found),
        "citation_recall_pass": expected.issubset(found),
        "citation_context_precision_pass": found.issubset(allowed),
        "json_valid": json_valid,
        "prediction": prediction,
    })
    if position % 10 == 0:
        print(f"Generated {position}/{sample_count}")

json_results = [row for row in results if row["json_valid"] is not None]
generation_metrics = {
    "num_examples": len(results),
    "citation_recall_pass_rate": sum(row["citation_recall_pass"] for row in results) / len(results),
    "citation_context_precision_pass_rate": sum(row["citation_context_precision_pass"] for row in results) / len(results),
    "json_valid_rate": (sum(row["json_valid"] for row in json_results) / len(json_results)) if json_results else None,
}

with (OUTPUT_DIR / "generation_evaluation.jsonl").open("w", encoding="utf-8") as handle:
    for row in results:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
(OUTPUT_DIR / "generation_metrics.json").write_text(json.dumps(generation_metrics, indent=2) + "\n", encoding="utf-8")
generation_metrics

## 12. Optional: save merged weights and upload the adapter

The LoRA adapter is sufficient for vLLM when LoRA serving is enabled. Merged 16-bit weights consume substantially more disk space, so merging is disabled by default.

In [ ]:
if SAVE_MERGED_16BIT:
    merged_dir = OUTPUT_DIR / "merged_16bit"
    model.save_pretrained_merged(str(merged_dir), tokenizer, save_method="merged_16bit")
    print("Saved merged model to", merged_dir)

if HF_MODEL_ID:
    token = os.getenv("HF_TOKEN")
    if not token:
        raise RuntimeError("HF_MODEL_ID is set, but HF_TOKEN is missing from the RunPod environment.")
    model.push_to_hub(HF_MODEL_ID, token=token)
    tokenizer.push_to_hub(HF_MODEL_ID, token=token)
    print("Uploaded LoRA adapter to", HF_MODEL_ID)
else:
    print("Hugging Face upload skipped; set HF_MODEL_ID and HF_TOKEN to enable it.")

## 13. Package the artifacts

Download the zip or keep it on the persistent `/workspace` volume before stopping the Pod.

In [ ]:
archive_path = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Artifact archive:", archive_path)
print("Remember to stop the RunPod Pod when training is complete.")